In [ ]:
pip install opencv-python

In [2]:
import os
import cv2
import numpy as np

In [4]:
dataset_paths_real = []

folder_real = r"E:\dataset\Deepfake Detection Dataset\4. validation_Dataset\real"

for real in os.listdir(folder_real):
    path_real = os.path.join(folder_real,real)
    dataset_paths_real.append(path_real)

display(dataset_paths_real)

['E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0000.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0001.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0002.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0003.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0004.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0005.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0006.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0007.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0008.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id0_0009.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\real\\id10_0000.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\

In [6]:
dataset_paths_fake = []

folder_fake = r"E:\dataset\Deepfake Detection Dataset\4. validation_Dataset\fake"

for fake in os.listdir(folder_fake):
    path_fake = os.path.join(folder_fake,fake)
    dataset_paths_fake.append(path_fake)

display(dataset_paths_fake)

['E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_02__meeting_serious__YVGY8LOK.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_02__outside_talking_still_laughing__YVGY8LOK.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_02__talking_against_wall__YVGY8LOK.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_02__walking_down_indoor_hall_disgust__YVGY8LOK.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_02__walk_down_hall_angry__YVGY8LOK.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_03__hugging_happy__ISF9SP4G.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_03__kitchen_pan__JZUXXFRB.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\\01_04__walk_down_hall_angry__GBC7ZGDP.mp4',
 'E:\\dataset\\Deepfake Detection Dataset\\4. validation_Dataset\\fake\

In [8]:
#display(dataset_paths_real,dataset_paths_fake)

In [10]:
print("Real videos:", len(dataset_paths_real))
print("Fake videos:", len(dataset_paths_fake))
print("Total videos:", len(dataset_paths_real) + len(dataset_paths_fake))

Real videos: 500
Fake videos: 500
Total videos: 1000


In [12]:
import os
import cv2
from tqdm import tqdm

# ── Paths ────────────────────────────────────────────────────
folder_real = r"E:\dataset\Deepfake Detection Dataset\4. validation_Dataset\real"
folder_fake = r"E:\dataset\Deepfake Detection Dataset\4. validation_Dataset\fake"
output_base = r"E:\dataset\Deepfake Detection Dataset\5. Validation_Dataset_Frames"

# ── Config ───────────────────────────────────────────────────
FRAMES_TO_EXTRACT = 20

# ── Already have these from your existing code ───────────────
dataset_paths_real = []
for real in os.listdir(folder_real):
    dataset_paths_real.append(os.path.join(folder_real, real))

dataset_paths_fake = []
for fake in os.listdir(folder_fake):
    dataset_paths_fake.append(os.path.join(folder_fake, fake))


def extract_frames(video_path, output_folder, n_frames=20):
    """Extract n evenly-spaced frames from a video and save as JPEGs."""
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"  [SKIP] Could not open: {video_path}")
        return False
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < n_frames:
        print(f"  [WARN] Only {total_frames} frames in {os.path.basename(video_path)}, extracting all.")
        indices = list(range(total_frames))
    else:
        # Evenly spaced indices across full video duration
        step = total_frames / n_frames
        indices = [int(step * i + step / 2) for i in range(n_frames)]
    
    os.makedirs(output_folder, exist_ok=True)
    
    saved = 0
    for frame_num in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        if ret:
            frame_path = os.path.join(output_folder, f"frame_{saved+1:02d}.jpg")
            cv2.imwrite(frame_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
    
    cap.release()
    return True


def process_split(video_paths, label, n_frames=20):
    """Process all videos for a given label (real or fake)."""
    print(f"\nProcessing {label.upper()} videos ({len(video_paths)} total)...")
    
    success, skipped = 0, 0
    
    for video_path in tqdm(video_paths, desc=label):
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_folder = os.path.join(output_base, label, video_name)
        
        # Skip if already extracted (resume support)
        if os.path.exists(output_folder):
            existing = len(os.listdir(output_folder))
            if existing == n_frames:
                skipped += 1
                continue
        
        result = extract_frames(video_path, output_folder, n_frames)
        if result:
            success += 1
    
    print(f"  Done — Extracted: {success} | Skipped (already exists): {skipped}")


# ── Run ──────────────────────────────────────────────────────
process_split(dataset_paths_real, "real", FRAMES_TO_EXTRACT)
process_split(dataset_paths_fake, "fake", FRAMES_TO_EXTRACT)

print("\nFrame extraction complete.")
print(f"Output saved to: {output_base}")


Processing REAL videos (500 total)...


real: 100%|███████████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 10868.66it/s]


  [WARN] Only 1 frames in id27_0005.mp4, extracting all.
  Done — Extracted: 1 | Skipped (already exists): 499

Processing FAKE videos (500 total)...


fake:   0%|                                                                                    | 0/500 [00:00<?, ?it/s]

  [WARN] Only 18 frames in 03_02__secret_conversation__GYX5OFTD.mp4, extracting all.


fake:  52%|██████████████████████████████████████▋                                   | 261/500 [06:15<07:58,  2.00s/it]

  [WARN] Only 5 frames in 12_20__secret_conversation__B0X1CGG2.mp4, extracting all.


fake: 100%|██████████████████████████████████████████████████████████████████████████| 500/500 [16:06<00:00,  1.93s/it]

  Done — Extracted: 396 | Skipped (already exists): 104

Frame extraction complete.
Output saved to: E:\dataset\Deepfake Detection Dataset\5. Validation_Dataset_Frames
